In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow version:", tf.__version__)

# Load & preprocess
df = pd.read_csv(r'C:\Users\Nicky\churn_prediction\data\healthcare\train_data.csv').copy()
df.drop(['case_id', 'patientid'], axis=1, inplace=True)
df['Bed Grade'] = df['Bed Grade'].fillna(df['Bed Grade'].mode()[0])
df['City_Code_Patient'] = df['City_Code_Patient'].fillna(df['City_Code_Patient'].mode()[0])

le = LabelEncoder()
cat_cols = [col for col in df.select_dtypes(include=['object', 'str']).columns if col != 'Stay']
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
df['Stay'] = le.fit_transform(df['Stay'].astype(str))

X = df.drop('Stay', axis=1)
y = df['Stay']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

num_classes = y.nunique()
print("✅ Data ready! Classes:", num_classes)

TensorFlow version: 2.21.0
✅ Data ready! Classes: 11


In [2]:
model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),

    layers.Dense(64, activation='relu'),
    layers.Dropout(0.2),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

C:\Users\Nicky\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 256)                 │           4,096 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 11)                  │             715 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 47,499 (185.54 KB)

 Trainable params: 46,731 (182.54 KB)

 Non-trainable params: 768 (3.00 KB)

In [3]:
#Train the Model 
history = model.fit(
    X_train_scaled, y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.1,
    verbose=1
)
print("✅ Training complete!")

Epoch 1/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.3383 - loss: 1.7692 - val_accuracy: 0.3906 - val_loss: 1.5900
Epoch 2/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.3829 - loss: 1.6181 - val_accuracy: 0.3990 - val_loss: 1.5643
Epoch 3/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.3910 - loss: 1.5934 - val_accuracy: 0.4055 - val_loss: 1.5533
Epoch 4/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.3961 - loss: 1.5792 - val_accuracy: 0.4038 - val_loss: 1.5460
Epoch 5/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.3987 - loss: 1.5711 - val_accuracy: 0.4077 - val_loss: 1.5423
Epoch 6/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4016 - loss: 1.5657 - val_accuracy: 0.4086 - val_loss: 1.5398
Epoch 7/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.4037 - loss: 1.5595 - val_accuracy: 0.4117 - val_loss: 1.5339
Epoch 8/20
448/448 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.4046 - loss: 1.5567 - val_accu

In [4]:
##Evaluate
loss, accuracy = model.evaluate(X_test_scaled, y_test, verbose=0)
print("🧠 Deep Learning Accuracy:", round(accuracy * 100, 2), "%")

dl_pred = model.predict(X_test_scaled).argmax(axis=1)
print(classification_report(y_test, dl_pred, zero_division=0))

🧠 Deep Learning Accuracy: 41.95 %
1991/1991 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step
              precision    recall  f1-score   support

           0       0.37      0.18      0.24      4689
           1       0.42      0.50      0.46     15561
           2       0.42      0.64      0.51     17603
           3       0.41      0.24      0.30     10981
           4       0.00      0.00      0.00      2357
           5       0.41      0.48      0.44      7128
           6       0.00      0.00      0.00       554
           7       0.00      0.00      0.00      2031
           8       0.33      0.22      0.27       941
           9       0.00      0.00      0.00       552
          10       0.50      0.37      0.43      1291

    accuracy                           0.42     63688
   macro avg       0.26      0.24      0.24     63688
weighted avg       0.38      0.42      0.39     63688



In [5]:
model.save(r'C:\Users\Nicky\churn_prediction\models\deep_learning_model.keras')
print("✅ Model saved!")

✅ Model saved!
